# Campus mobility prediction
Select the `.venv` Python kernel and Run All from this project folder.
Five readings predict the serving cell three seconds ahead. Outputs go to `artifacts/`.


In [1]:
import copy
import csv
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

seed = 42
hidden_size = 32
pretrain_epochs = 20
adapt_epochs = 40
batch_size = 512
learning_rate = 0.001
patience = 8
data_dir = Path("data")
output_dir = Path("artifacts")

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
data_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)


## Generate campus trips
Person 1's simulator uses four base stations, noisy positions, and fixed seeds for five
separate trip sets. Running the next cells regenerates the CSVs.


In [2]:
PLACES = {
    "hostel": (50, 50),
    "library": (350, 80),
    "lecture_a": (230, 290),
    "lecture_b": (340, 330),
    "canteen": (60, 300),
}

# A station's position in this list is its cell ID, from 0 to 3.
BASE_STATIONS = [(100, 100), (300, 100), (100, 300), (300, 300)]
NOISE_M = 3.0

SPLITS = [
    # File name, scenario, number of trips, random seed
    ("general_train", "general", 300, 1),
    ("general_val", "general", 50, 2),
    ("morning_adapt", "morning", 50, 3),
    ("morning_val", "morning", 30, 4),
    ("morning_test", "morning", 100, 5),
]


In [3]:
def serving_cell(x, y):
    closest_cell = 0
    shortest_distance = float("inf")

    for cell_id, (station_x, station_y) in enumerate(BASE_STATIONS):
        distance = (x - station_x) ** 2 + (y - station_y) ** 2
        if distance < shortest_distance:
            shortest_distance = distance
            closest_cell = cell_id

    return closest_cell


def pick_stops(scenario, rng):
    if scenario == "morning":
        destinations = ["lecture_a", "lecture_b", "library", "canteen"]
        weights = [4, 4, 1, 1]
        destination = rng.choices(destinations, weights=weights)[0]
        return ["hostel", destination]

    place_names = list(PLACES)
    stops = [rng.choice(place_names)]
    for leg in range(3):
        possible_stops = []
        for name in place_names:
            if name != stops[-1]:
                possible_stops.append(name)
        stops.append(rng.choice(possible_stops))

    return stops


In [4]:
def make_trip(trip_id, scenario, rng, dt=1.0):
    stops = pick_stops(scenario, rng)
    walking_speed = rng.uniform(1.0, 1.6)
    x, y = PLACES[stops[0]]
    heading = 0.0
    readings = []

    def record_reading(speed):
        # Record the current position. Only the observed position gets noise.
        time = round(len(readings) * dt, 3)
        observed_x = round(x + rng.gauss(0, NOISE_M), 2)
        observed_y = round(y + rng.gauss(0, NOISE_M), 2)
        cell = serving_cell(x, y)
        readings.append([trip_id, time, observed_x, observed_y,
                         speed, round(heading, 3), cell])

    for stop_index in range(1, len(stops)):
        destination_x, destination_y = PLACES[stops[stop_index]]

        # Add a point near the middle to give each route some variation.
        sideways_angle = math.atan2(destination_y - y, destination_x - x) + math.pi / 2
        offset = rng.uniform(-40, 40)
        middle_x = (x + destination_x) / 2 + offset * math.cos(sideways_angle)
        middle_y = (y + destination_y) / 2 + offset * math.sin(sideways_angle)
        route = [(middle_x, middle_y), (destination_x, destination_y)]

        for target_x, target_y in route:
            heading = math.atan2(target_y - y, target_x - x)
            while math.hypot(target_x - x, target_y - y) > walking_speed * dt:
                record_reading(round(walking_speed, 2))
                x += walking_speed * dt * math.cos(heading)
                y += walking_speed * dt * math.sin(heading)
            x, y = target_x, target_y

        # Pause at intermediate stops, then continue walking.
        if stop_index < len(stops) - 1:
            pause_seconds = rng.randint(0, 20)
            for pause_step in range(int(pause_seconds / dt)):
                record_reading(0.0)

    record_reading(0.0)
    return readings


In [5]:
columns = ["trip_id", "time_s", "x_m", "y_m", "speed_m_s", "heading_rad", "serving_cell"]

for name, scenario, trip_count, split_seed in SPLITS:
    rng = random.Random(split_seed)
    csv_path = data_dir / f"{name}.csv"

    with csv_path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(columns)

        for trip_id in range(trip_count):
            trip_readings = make_trip(trip_id, scenario, rng)
            writer.writerows(trip_readings)


## Read and check trips
IDs are local to each file. Check one-second timestamps, valid cell IDs, and duplicate trips.
Handovers have one-second resolution. Matching sub-second events and the map demo remain
pending in Person 1's work.


In [6]:
def read_trips(path):
    trips = {}

    with path.open(newline="") as file:
        reader = csv.reader(file)
        header = next(reader)
        assert header == columns, f"Unexpected columns in {path}"

        for row in reader:
            trip_id = int(row[0])
            reading = []
            for value in row[1:]:
                reading.append(float(value))

            if trip_id not in trips:
                trips[trip_id] = []
            trips[trip_id].append(reading)

    for trip_id, readings in trips.items():
        readings = np.asarray(readings, dtype=np.float32)
        assert readings.shape[1] == 6 and len(readings) >= 8
        assert np.isfinite(readings).all()

        times = readings[:, 0]
        cells = readings[:, 5]
        assert np.array_equal(times, np.arange(len(readings)))
        assert np.isin(cells, [0, 1, 2, 3]).all()
        trips[trip_id] = readings

    return trips


In [7]:
data = {}
seen_trips = set()
for name, scenario, trip_count, split_seed in SPLITS:
    csv_path = data_dir / f"{name}.csv"
    trips = read_trips(csv_path)
    assert sorted(trips) == list(range(trip_count))

    for readings in trips.values():
        # Compare full trip contents because IDs restart in each file.
        trip_contents = readings.tobytes()
        assert trip_contents not in seen_trips, "Duplicate trip"
        seen_trips.add(trip_contents)

    data[name] = trips

print("All five datasets passed the checks.")


All five datasets passed the checks.


## Make training windows
Each input contains five readings from one trip. Its label is the cell three seconds
after the last reading. For example, seconds 8–12 predict the cell at second 15.

After removing the trip ID, each reading has six columns. In the code, `readings[:, 1]`
means take column 1 from every row.

| Column | Value |
|---|---|
| 0 | Time |
| 1, 2 | Position x, y |
| 3 | Speed |
| 4 | Heading |
| 5 | Serving cell |

`make_windows` returns three items, the input windows, future cell labels, and details
containing the trip ID, last input time, and current cell for each window.

A tensor is a PyTorch array. The input shape is (number of windows, 5 readings, 9 features).


In [8]:
def encode(readings):
    # Each reading becomes nine input values for the GRU.
    features = np.zeros((len(readings), 9), dtype=np.float32)
    features[:, 0] = readings[:, 1] / 400  # x position, scaled to campus size
    features[:, 1] = readings[:, 2] / 400  # y position
    features[:, 2] = readings[:, 3] / 1.6  # speed, scaled to walking speed
    features[:, 3] = np.sin(readings[:, 4])
    features[:, 4] = np.cos(readings[:, 4])

    # Four flags represent the cell. Cell 2 becomes [0, 0, 1, 0].
    for row_index in range(len(readings)):
        cell_id = int(readings[row_index, 5])
        features[row_index, 5 + cell_id] = 1

    return features


def make_windows(trips):
    input_windows = []
    future_cells = []
    window_details = []

    for trip_id, readings in sorted(trips.items()):
        features = encode(readings)

        # Start at the fifth reading. Leave three readings for the future label.
        for reading_index in range(4, len(readings) - 3):
            window_start = reading_index - 4
            window_stop = reading_index + 1
            future_index = reading_index + 3

            input_windows.append(features[window_start:window_stop])
            future_cells.append(int(readings[future_index, 5]))

            time = int(readings[reading_index, 0])
            current_cell = int(readings[reading_index, 5])
            window_details.append([trip_id, time, current_cell])

    inputs = torch.from_numpy(np.stack(input_windows))
    labels = torch.tensor(future_cells)
    details = np.asarray(window_details)
    return inputs, labels, details


In [9]:
general_train = make_windows(data["general_train"])
general_val = make_windows(data["general_val"])
morning_val = make_windows(data["morning_val"])


In [10]:
# Check windows using two short trips with a known cell change at second 7.
sample_trips = {}
for trip_id in range(2):
    readings = []
    for time in range(12):
        cell = 0
        if time >= 7:
            cell = 1
        x = 100 * trip_id + time
        readings.append([time, x, 20, 1.2, 0, cell])
    sample_trips[trip_id] = np.asarray(readings, dtype=np.float32)

inputs, labels, details = make_windows(sample_trips)
assert inputs.shape == (10, 5, 9)
for window_index, (trip_id, time, current_cell) in enumerate(details):
    trip = sample_trips[trip_id]
    expected_label = trip[time + 3, 5]
    expected_inputs = encode(trip[time - 4:time + 1])
    assert labels[window_index] == expected_label
    np.testing.assert_array_equal(inputs[window_index], expected_inputs)


## Train the GRU
A 32-unit GRU outputs four cell scores. Training uses Adam and cross-entropy.
Keep the weights with the lowest validation loss and stop after eight unimproved epochs.

An epoch visits all training windows in batches of 512. Each batch updates the weights.
Validation measures prediction errors on separate trips. Lower validation loss is better.


In [11]:
class MobilityGRU(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.gru = nn.GRU(9, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 4)

    def forward(self, inputs):
        sequence_output, hidden_state = self.gru(inputs)
        last_reading_state = hidden_state[0]  # The model has one GRU layer.
        cell_scores = self.classifier(last_reading_state)
        return cell_scores


@torch.inference_mode()
def predict(model, inputs):
    model.eval()
    all_probabilities = []

    for batch in inputs.split(2048):
        cell_scores = model(batch)
        cell_probabilities = torch.softmax(cell_scores, dim=1)
        all_probabilities.append(cell_probabilities.numpy())

    return np.concatenate(all_probabilities)


In [12]:
def fit(model, train, validation, epochs):
    torch.manual_seed(seed)
    training_inputs, training_labels, training_details = train
    validation_inputs, validation_labels, validation_details = validation

    training_dataset = TensorDataset(training_inputs, training_labels)
    shuffle_generator = torch.Generator().manual_seed(seed)
    batches = DataLoader(training_dataset, batch_size=batch_size,
                         shuffle=True, generator=shuffle_generator)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    best_loss = float("inf")
    best_epoch = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch_inputs, batch_labels in batches:
            # Predict, measure the error, then update the weights.
            optimizer.zero_grad()
            cell_scores = model(batch_inputs)
            loss = loss_function(cell_scores, batch_labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Check how much probability the model gave each correct validation cell.
        probabilities = predict(model, validation_inputs)
        window_numbers = np.arange(len(validation_labels))
        correct_cells = validation_labels.numpy()
        # For window 0 with true cell 2, this selects probabilities[0, 2].
        correct_cell_probabilities = probabilities[window_numbers, correct_cells]
        safe_probabilities = np.maximum(correct_cell_probabilities, 1e-12)
        validation_loss = -np.log(safe_probabilities).mean()

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())

        epochs_without_improvement = epoch - best_epoch
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_weights)
    print(f"Best epoch {best_epoch}, validation loss {best_loss:.4f}")


In [13]:
def save_model(model, name):
    checkpoint = {
        "state_dict": model.state_dict(),
        "hidden_size": model.gru.hidden_size,
    }
    checkpoint_path = output_dir / f"{name}.pt"
    torch.save(checkpoint, checkpoint_path)


def load_model(name):
    checkpoint_path = output_dir / f"{name}.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = MobilityGRU(checkpoint["hidden_size"])
    model.load_state_dict(checkpoint["state_dict"])
    return model


## Pre-train, fine-tune, and compare with scratch training
Pre-training uses general trips. Each fine-tuned model starts from the same pre-trained
weights. Fine-tuning and scratch training use the same first 10, 25, or 50 morning trips.
Validation trips select checkpoints. Test trips are evaluated after training.


In [14]:
torch.manual_seed(seed)
pretrained = MobilityGRU(hidden_size)
fit(pretrained, general_train, general_val, pretrain_epochs)
save_model(pretrained, "pretrained")
model_names = ["pretrained"]


Best epoch 19, validation loss 0.0222


In [15]:
for trip_count in (10, 25, 50):
    adaptation_trips = {}
    for trip_id in range(trip_count):
        adaptation_trips[trip_id] = data["morning_adapt"][trip_id]
    adaptation_data = make_windows(adaptation_trips)

    for method in ("finetuned", "scratch"):
        model_name = f"{method}_{trip_count}"
        print("Training", model_name, flush=True)
        torch.manual_seed(seed)
        model = MobilityGRU(hidden_size)

        if method == "finetuned":
            model.load_state_dict(pretrained.state_dict())

        fit(model, adaptation_data, morning_val, adapt_epochs)
        save_model(model, model_name)
        model_names.append(model_name)


Training finetuned_10


Best epoch 12, validation loss 0.0249
Training scratch_10


Best epoch 40, validation loss 0.1343
Training finetuned_25


Best epoch 2, validation loss 0.0267
Training scratch_25


Best epoch 40, validation loss 0.0771
Training finetuned_50


Best epoch 3, validation loss 0.0254
Training scratch_50


Best epoch 40, validation loss 0.0467


## Evaluate and export predictions
All models use the same held-out windows. Changed-target accuracy measures windows where
the future cell differs from the current cell. Each exported row gives four probabilities
in cell-ID order, ready for Persons 3 and 4.


In [16]:
test_inputs, test_labels, test_details = make_windows(data["morning_test"])
future_cells = test_labels.numpy()
current_cells = test_details[:, 2]
cell_changes = future_cells != current_cells
comparison = {}

print(f"{'Model':22} {'Overall':>8} {'Changed':>8}")
for model_name in model_names:
    model = load_model(model_name)
    probabilities = predict(model, test_inputs)

    assert probabilities.shape == (len(test_labels), 4)
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)

    predicted_cells = probabilities.argmax(axis=1)
    correct_predictions = predicted_cells == future_cells
    overall_accuracy = float(correct_predictions.mean())
    changed_cell_accuracy = float(correct_predictions[cell_changes].mean())
    comparison[model_name] = {
        "accuracy": overall_accuracy,
        "changed_accuracy": changed_cell_accuracy,
    }
    print(f"{model_name:22} {overall_accuracy:8.1%} {changed_cell_accuracy:8.1%}")

    prediction_path = output_dir / f"{model_name}_predictions.csv"
    with prediction_path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["trip_id", "time_s", "serving_cell", "target_time_s",
                         "p_cell_0", "p_cell_1", "p_cell_2", "p_cell_3"])

        for window_index, (trip_id, time, current_cell) in enumerate(test_details):
            target_time = time + 3
            row = [int(trip_id), float(time), int(current_cell), float(target_time)]
            row.extend(probabilities[window_index].tolist())
            writer.writerow(row)


Model                   Overall  Changed


pretrained                98.8%    76.3%


finetuned_10              98.9%    68.2%


scratch_10                97.2%     0.0%


finetuned_25              98.6%    64.2%


scratch_25                97.7%     0.0%


finetuned_50              98.8%    55.9%


scratch_50                98.2%    30.1%


In [17]:
# A simple reference predicts that the user stays in the current cell.
comparison["current_cell_baseline"] = {
    "accuracy": float((current_cells == future_cells).mean()),
    "changed_accuracy": 0.0,
}

settings = {
    "seed": seed,
    "hidden_size": hidden_size,
    "pretrain_epochs": pretrain_epochs,
    "adapt_epochs": adapt_epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "patience": patience,
}
report = {"config": settings, "test": comparison}
results_path = output_dir / "results.json"
results_path.write_text(json.dumps(report, indent=2), encoding="utf-8")


1048

## Demo on one held-out trip
Change the trip ID and adaptation budget, then rerun this cell. It shows every window
where the cell changes within three seconds, including incorrect predictions.


In [18]:
trip_id = 0
adaptation_budget = 10

demo_trip = {trip_id: data["morning_test"][trip_id]}
demo_inputs, demo_labels, demo_details = make_windows(demo_trip)
demo_names = ["pretrained", f"scratch_{adaptation_budget}", f"finetuned_{adaptation_budget}"]

demo_predictions = {}
for model_name in demo_names:
    model = load_model(model_name)
    demo_predictions[model_name] = predict(model, demo_inputs)

for window_index, (trip_id, time, current_cell) in enumerate(demo_details):
    future_cell = int(demo_labels[window_index])
    if future_cell == current_cell:
        continue

    print("Time", time, "current cell", current_cell, "future cell", future_cell)
    for model_name in demo_names:
        probabilities = demo_predictions[model_name][window_index]
        predicted_cell = int(probabilities.argmax())
        future_cell_probability = round(float(probabilities[future_cell]), 3)
        print(model_name, "predicts", predicted_cell,
              "with probability", future_cell_probability, "assigned to the true future cell")
    print()


Time 132 current cell 0 future cell 2
pretrained predicts 2 with probability 0.863 assigned to the true future cell
scratch_10 predicts 0 with probability 0.085 assigned to the true future cell
finetuned_10 predicts 2 with probability 0.761 assigned to the true future cell

Time 133 current cell 0 future cell 2
pretrained predicts 2 with probability 0.983 assigned to the true future cell
scratch_10 predicts 0 with probability 0.089 assigned to the true future cell
finetuned_10 predicts 2 with probability 0.973 assigned to the true future cell

Time 134 current cell 0 future cell 2
pretrained predicts 2 with probability 0.961 assigned to the true future cell
scratch_10 predicts 0 with probability 0.086 assigned to the true future cell
finetuned_10 predicts 2 with probability 0.933 assigned to the true future cell

Time 156 current cell 2 future cell 3
pretrained predicts 3 with probability 0.574 assigned to the true future cell
scratch_10 predicts 2 with probability 0.115 assigned to th